# Aula 006 — Algoritmo PRISM
## Geração da 1ª regra para a classe `soft` — Dataset Lentes de Contato (24 exemplos)

In [ ]:
import pandas as pd

# ── Dataset completo de Lentes de Contato (24 exemplos) ──
dados = pd.DataFrame({
    'age':            ['young']*8 + ['pre-presbyopic']*8 + ['presbyopic']*8,
    'spectacle':      ['myope','myope','myope','myope',
                       'hypermetrope','hypermetrope','hypermetrope','hypermetrope']*3,
    'astigmatism':    ['no','no','yes','yes','no','no','yes','yes']*3,
    'tear':           ['reduced','normal','reduced','normal',
                       'reduced','normal','reduced','normal']*3,
    'lenses':         ['none','soft','none','hard',
                       'none','soft','none','hard',
                       'none','soft','none','hard',
                       'none','soft','none','none',
                       'none','none','none','hard',
                       'none','soft','none','none']
})

print(f"Total de exemplos: {len(dados)}")
print(f"Distribuição das classes:")
print(dados['lenses'].value_counts().to_string())
print()
display(dados)

---
## Algoritmo PRISM — Pseudocódigo

Para a classe alvo C:
1. Comece com antecedente vazio: `IF <vazio> THEN lenses = C`
2. Para cada par atributo=valor ainda não usado na regra, calcule `p(C | atributo=valor)` no subconjunto atual
3. Escolha o par que **maximiza** essa probabilidade (desempate: maior cobertura positiva)
4. Adicione o par à regra e filtre o subconjunto
5. Repita até a regra ser **perfeita** (100% de precisão no subconjunto)
6. Remova os exemplos cobertos e gere a próxima regra

**Neste exercício: apenas a 1ª regra para `soft`.**

In [ ]:
# ── Função auxiliar: calcula p(classe_alvo | attr=val) para todos os pares ──
def calcular_probabilidades(df, classe_alvo, atributos_disponiveis):
    """Retorna DataFrame com p(classe|attr=val) para cada par atributo-valor."""
    resultados = []
    for attr in atributos_disponiveis:
        for val in df[attr].unique():
            sub = df[df[attr] == val]
            total = len(sub)
            positivos = len(sub[sub['lenses'] == classe_alvo])
            prob = positivos / total if total > 0 else 0
            resultados.append({
                'atributo = valor': f'{attr} = {val}',
                f'p({classe_alvo} | attr=val)': f'{positivos}/{total} = {prob:.2f}',
                'cobertura positiva': positivos,
                'prob': prob  # para ordenação
            })
    
    resultado_df = pd.DataFrame(resultados)
    resultado_df = resultado_df.sort_values(
        by=['prob', 'cobertura positiva'], ascending=[False, False]
    ).reset_index(drop=True)
    return resultado_df

---
## Regra 1 para classe `soft`

### Passo 1 — Antecedente vazio
Partimos do conjunto completo S (24 exemplos) e calculamos `p(soft | atributo=valor)` para todos os 9 pares possíveis.

In [ ]:
classe_alvo = 'soft'
atributos = ['age', 'spectacle', 'astigmatism', 'tear']

# Passo 1: conjunto completo, antecedente vazio
S = dados.copy()
atributos_disponiveis = atributos.copy()
antecedente = []

print(f"Conjunto atual: {len(S)} exemplos ({len(S[S['lenses']==classe_alvo])} da classe '{classe_alvo}')")
print(f"Antecedente atual: IF <vazio> THEN lenses = {classe_alvo}")
print()

tabela1 = calcular_probabilidades(S, classe_alvo, atributos_disponiveis)
display(tabela1[['atributo = valor', f'p({classe_alvo} | attr=val)', 'cobertura positiva']])

# Melhor escolha
melhor = tabela1.iloc[0]
print(f"\n>>> Escolha do passo 1: {melhor['atributo = valor']}  ({melhor[f'p({classe_alvo} | attr=val)']})")

### Passo 2 — Refinamento
Filtramos o subconjunto com a condição escolhida e repetimos o cálculo com os atributos restantes.

In [ ]:
# Aplicar a escolha do passo 1
# tear = normal foi a melhor (ou astigmatism = no, dependendo do desempate)
# Vamos seguir o critério: maior p, desempate por maior cobertura positiva

melhor_par = tabela1.iloc[0]['atributo = valor']
attr_escolhido, val_escolhido = melhor_par.split(' = ')
antecedente.append(melhor_par)
S = S[S[attr_escolhido] == val_escolhido].copy()
atributos_disponiveis.remove(attr_escolhido)

n_pos = len(S[S['lenses'] == classe_alvo])
n_total = len(S)
print(f"Antecedente atual: IF {' AND '.join(antecedente)} THEN lenses = {classe_alvo}")
print(f"Subconjunto: {n_total} exemplos ({n_pos} da classe '{classe_alvo}')")
print(f"Precisão atual: {n_pos}/{n_total} = {n_pos/n_total:.2f}")
print()

if n_pos == n_total:
    print(">>> REGRA PERFEITA! Paramos aqui.")
else:
    tabela2 = calcular_probabilidades(S, classe_alvo, atributos_disponiveis)
    display(tabela2[['atributo = valor', f'p({classe_alvo} | attr=val)', 'cobertura positiva']])
    melhor2 = tabela2.iloc[0]
    print(f"\n>>> Escolha do passo 2: {melhor2['atributo = valor']}  ({melhor2[f'p({classe_alvo} | attr=val)']})")

In [ ]:
# Passo 3 — Novo refinamento (se necessário)
melhor_par2 = tabela2.iloc[0]['atributo = valor']
attr2, val2 = melhor_par2.split(' = ')
antecedente.append(melhor_par2)
S = S[S[attr2] == val2].copy()
atributos_disponiveis.remove(attr2)

n_pos = len(S[S['lenses'] == classe_alvo])
n_total = len(S)
print(f"Antecedente atual: IF {' AND '.join(antecedente)} THEN lenses = {classe_alvo}")
print(f"Subconjunto: {n_total} exemplos ({n_pos} da classe '{classe_alvo}')")
print(f"Precisão atual: {n_pos}/{n_total} = {n_pos/n_total:.2f}")
print()

if n_pos == n_total:
    print(">>> REGRA PERFEITA! Paramos aqui.")
else:
    tabela3 = calcular_probabilidades(S, classe_alvo, atributos_disponiveis)
    display(tabela3[['atributo = valor', f'p({classe_alvo} | attr=val)', 'cobertura positiva']])
    melhor3 = tabela3.iloc[0]
    print(f"\n>>> Escolha do passo 3: {melhor3['atributo = valor']}  ({melhor3[f'p({classe_alvo} | attr=val)']})")

In [ ]:
# Passo 4 — Mais um refinamento se ainda não for perfeita
if n_pos != n_total:
    melhor_par3 = tabela3.iloc[0]['atributo = valor']
    attr3, val3 = melhor_par3.split(' = ')
    antecedente.append(melhor_par3)
    S = S[S[attr3] == val3].copy()
    atributos_disponiveis.remove(attr3)

    n_pos = len(S[S['lenses'] == classe_alvo])
    n_total = len(S)
    print(f"Antecedente atual: IF {' AND '.join(antecedente)} THEN lenses = {classe_alvo}")
    print(f"Subconjunto: {n_total} exemplos ({n_pos} da classe '{classe_alvo}')")
    print(f"Precisão atual: {n_pos}/{n_total} = {n_pos/n_total:.2f}")
    print()
    if n_pos == n_total:
        print(">>> REGRA PERFEITA! Paramos aqui.")
    else:
        print("Não há mais atributos disponíveis. Regra final (não perfeita).")

---
## Resultado Final

In [ ]:
print("="*70)
print("1ª REGRA GERADA PELO PRISM PARA A CLASSE 'soft'")
print("="*70)
print(f"\n  IF {' AND '.join(antecedente)}")
print(f"  THEN lenses = {classe_alvo}")
print(f"\n  Cobertura: {n_pos} exemplos da classe '{classe_alvo}'")
print(f"  Precisão: {n_pos}/{n_total} = {n_pos/n_total:.2%}")
print()
print("Exemplos cobertos pela regra:")
display(S)